# Semana 3 — Lectura de datos y mínimos cuadrados en Fortran
**Física Computacional — 106018C** · Lenguaje: **Fortran 2008**

En este notebook aprenderemos a traducir a Fortran algunas estructuras que ya usamos en Python, leer mediciones desde un archivo y programar un ajuste lineal por mínimos cuadrados. Al final aplicaremos todo al experimento del péndulo simple.

## Objetivos de la sesión

- Recuperar las ideas de declaración de variables y precisión doble cuando sean necesarias.
- Escribir ciclos `do` y `do while` y compararlos con Python.
- Abrir y leer archivos de texto con `open`, `read`, `iostat` y `close`.
- Construir acumuladores para un ajuste lineal por mínimos cuadrados.
- Estimar la aceleración de la gravedad a partir de mediciones de un péndulo.
- Visualizar en Python los datos, la recta ajustada y los residuos.


## 0. Preparación de Google Colab

Colab ejecuta Python de manera nativa. Para trabajar con Fortran escribiremos cada programa en un archivo `.f90`, lo compilaremos con `gfortran` y ejecutaremos el archivo resultante. Ejecute primero esta celda.

In [ ]:
!which gfortran || (apt-get update -qq && apt-get install -y gfortran)
!gfortran --version | head -n 1

<div style="background-color:#eef7ee; border-left:5px solid #4caf50; padding:10px 15px; margin:10px 0;">
<b>Flujo de trabajo:</b> una celda con <code>%%writefile nombre.f90</code> guarda el código. La celda siguiente llama al compilador y luego ejecuta el programa. Si modifica el código, debe volver a ejecutar ambas celdas.
</div>

## 1. Ciclos: de Python a Fortran

En Python, `range(1, 6)` produce 1, 2, 3, 4 y 5. En Fortran, el límite superior del ciclo `do i = 1, 5` también se incluye. Los arreglos de Fortran suelen comenzar en 1.

```python
suma = 0
for i in range(1, 6):
    suma = suma + i
```

In [ ]:
%%writefile ciclos.f90
program ciclos
  implicit none                    ! Obliga a declarar todas las variables

  integer :: i, suma               ! i: contador; suma: acumulador

  suma = 0                         ! Todo acumulador debe inicializarse

  ! Recorre los enteros desde 1 hasta 5, incluyendo ambos extremos
  do i = 1, 5
    suma = suma + i                ! Actualiza el acumulador
    print *, 'i =', i, 'suma parcial =', suma  ! Muestra el avance
  end do

  print *, 'suma final =', suma    ! Resultado después del ciclo
end program ciclos

In [ ]:
!gfortran -std=f2008 -Wall -Wextra -fcheck=all -o ciclos ciclos.f90
!./ciclos

### Tres formas de ciclo

Fortran ofrece tres formas que usaremos en esta sesión:

    ! Número conocido de repeticiones
    do i = 1, n
      ! instrucciones
    end do

    ! Repetir mientras se cumpla una condición
    do while (condicion)
      ! instrucciones
    end do

    ! Número desconocido de repeticiones
    do
      if (condicion_de_salida) exit
    end do

La tercera forma será especialmente útil para leer un archivo cuyo número de filas no conocemos.

### Ejercicio 1 — Completar un ciclo

Complete los dos espacios `_____` para calcular $1^2+2^2+\cdots+10^2$. El resultado debe ser 385. Después compile y ejecute.

In [ ]:
%%writefile ejercicio_ciclo.f90
program ejercicio_ciclo
  implicit none                         ! Exige declarar las variables
  integer :: i, suma_cuadrados          ! Contador y acumulador

  suma_cuadrados = 0                   ! Inicializa el acumulador

  ! TODO: complete el límite superior y el término que se suma
  do i = 1, _____
    suma_cuadrados = suma_cuadrados + _____
  end do

  print *, 'resultado =', suma_cuadrados  ! Debe producir 385
end program ejercicio_ciclo

In [ ]:
# Esta celda producirá un error hasta que complete la anterior.
!gfortran -std=f2008 -Wall -Wextra -fcheck=all -o ejercicio_ciclo ejercicio_ciclo.f90
!./ejercicio_ciclo

<div style="background-color:#fdecea; border-left:5px solid #e53935; padding:10px 15px; margin:10px 0;">
<b>Cuidado:</b> Fortran no usa la indentación para delimitar un ciclo. Sin embargo, indentar el contenido permite reconocer la estructura. Todo ciclo debe cerrarse con <code>end do</code>.
</div>

## 2. Lectura de datos desde un archivo

El archivo que leerá Fortran será texto plano. Cada fila de `pendulo_limpio.dat` tendrá cinco columnas separadas por espacios:

```text
medicion_id  longitud_cm  angulo_inicial_deg  numero_oscilaciones  tiempo_medido_s
```

No tendrá encabezado, porque el programa espera números desde la primera línea. Creemos primero un archivo pequeño para practicar.

In [ ]:
%%writefile pendulo_demo.dat
1 40.0 5.0 10 12.72
2 55.0 8.0 10 14.98
3 70.0 6.0 10 16.80
4 85.0 7.0 10 18.50

In [ ]:
%%writefile leer_archivo.f90
program leer_archivo
  use iso_fortran_env, only: real64, iostat_end  ! Precisión y código de fin de archivo
  implicit none                                  ! Evita variables implícitas

  integer :: unidad, estado, id, n_osc, n_datos  ! Control del archivo y campos enteros
  real(real64) :: longitud_cm, angulo_deg, tiempo_s, periodo_s  ! Campos reales

  ! Abre un archivo existente únicamente para lectura
  open(newunit=unidad, file='pendulo_demo.dat', status='old', &
       action='read', iostat=estado)

  if (estado /= 0) error stop 'No fue posible abrir el archivo'  ! Verifica la apertura

  n_datos = 0                    ! Inicializa el contador de filas

  ! El número de filas es desconocido: se lee hasta encontrar el final
  do
    read(unidad, *, iostat=estado) id, longitud_cm, angulo_deg, &
         n_osc, tiempo_s

    if (estado == iostat_end) exit                    ! Fin normal del archivo
    if (estado /= 0) error stop 'Hay una fila mal formada'  ! Otro valor indica error

    periodo_s = tiempo_s / real(n_osc, real64)        ! Período de una oscilación
    n_datos = n_datos + 1                             ! Cuenta la fila aceptada
    print '(I3,3F12.4)', id, longitud_cm, tiempo_s, periodo_s  ! Muestra la fila
  end do

  close(unidad)                                      ! Libera el archivo
  print *, 'Número de datos leídos =', n_datos       ! Resumen de la lectura
end program leer_archivo

In [ ]:
!gfortran -std=f2008 -Wall -Wextra -fcheck=all -o leer_archivo leer_archivo.f90
!./leer_archivo

### ¿Qué hace cada instrucción?

- `newunit=unidad` solicita un identificador disponible para el archivo.
- `status='old'` exige que el archivo ya exista.
- `action='read'` indica que solamente se leerá.
- El asterisco en `read(unidad, *)` permite leer números separados por espacios.
- `iostat_end` identifica el final normal del archivo.
- Cualquier otro valor no nulo de `estado` indica un problema de lectura.
- `close` libera el archivo cuando termina la lectura.

Observe que no necesitamos conocer por anticipado el número de filas. El ciclo termina al alcanzar el final del archivo.

### Ejercicio 2 — Completar la lectura

Complete las tres partes marcadas con `_____`. El programa debe contar las mediciones con longitud mayor que 60 cm. Para el archivo de demostración debe obtener 2.

In [ ]:
%%writefile ejercicio_lectura.f90
program ejercicio_lectura
  use iso_fortran_env, only: real64, iostat_end  ! Precisión y fin de archivo
  implicit none                                  ! Exige declarar las variables

  integer :: unidad, estado, id, n_osc, contador  ! Variables enteras
  real(real64) :: longitud_cm, angulo_deg, tiempo_s  ! Variables leídas

  contador = 0                    ! Inicializa el número de casos encontrados

  ! Abre el archivo y comprueba que la operación funcionó
  open(newunit=unidad, file='pendulo_demo.dat', status='old', &
       action='read', iostat=estado)
  if (estado /= 0) error stop 'No fue posible abrir el archivo'

  ! TODO: complete la lista de variables, la condición de salida y el contador
  do
    read(unidad, *, iostat=estado) _____
    if (estado == _____) exit
    if (estado /= 0) error stop 'Error de lectura'

    if (longitud_cm > 60.0_real64) contador = _____
  end do

  close(unidad)                    ! Cierra el archivo al terminar
  print *, 'Mediciones con L > 60 cm =', contador  ! Debe producir 2
end program ejercicio_lectura

In [ ]:
# Esta celda producirá un error hasta que complete la anterior.
!gfortran -std=f2008 -Wall -Wextra -fcheck=all -o ejercicio_lectura ejercicio_lectura.f90
!./ejercicio_lectura

## 3. Mínimos cuadrados en una dimensión

Supongamos que medimos pares $(x_i,y_i)$ y proponemos el modelo lineal

$$\widehat y_i=ax_i+b.$$

El residuo de la medición $i$ es

$$r_i=y_i-\widehat y_i=y_i-(ax_i+b).$$

El método de mínimos cuadrados escoge $a$ y $b$ para minimizar

$$S(a,b)=\sum_{i=1}^{N}[y_i-(ax_i+b)]^2.$$

Al imponer $\partial S/\partial a=0$ y $\partial S/\partial b=0$ se obtiene

$$a=\frac{N\sum x_i y_i-(\sum x_i)(\sum y_i)}{N\sum x_i^2-(\sum x_i)^2},\qquad b=\frac{\sum y_i-a\sum x_i}{N}.$$

### Acumuladores

Solo necesitamos mantener cuatro sumas mientras leemos el archivo:

$$s_x=\sum x_i,\quad s_y=\sum y_i,\quad s_{xx}=\sum x_i^2,\quad s_{xy}=\sum x_iy_i.$$

Esto permite procesar el archivo fila por fila, sin guardar necesariamente todos los datos en arreglos. Antes del ciclo todos los acumuladores deben inicializarse en cero.

Para estudiar únicamente el algoritmo de ajuste, el siguiente ejemplo usa cinco pares guardados en arreglos. La lectura del archivo y el ajuste permanecen separados; más adelante deberá combinarlos.

In [ ]:
%%writefile datos_lineales.dat
0.0 1.1
1.0 2.9
2.0 5.2
3.0 6.8
4.0 9.1

In [ ]:
%%writefile ajuste_lineal.f90
program ajuste_lineal
  use iso_fortran_env, only: real64  ! Selecciona doble precisión
  implicit none                      ! Exige declarar todas las variables

  integer, parameter :: n = 5        ! Número de pares (x,y)
  real(real64) :: x(n), y(n)         ! Arreglos con los datos
  integer :: i                       ! Índice de los ciclos
  real(real64) :: sx, sy, sxx, sxy   ! Acumuladores del ajuste
  real(real64) :: den, a, b           ! Denominador, pendiente e intercepto
  real(real64) :: y_ajustada, sse, sst, r2  ! Variables para evaluar la calidad

  ! Datos pequeños para estudiar el algoritmo sin mezclarlo con la lectura
  x = [0.0_real64, 1.0_real64, 2.0_real64, 3.0_real64, 4.0_real64]
  y = [1.1_real64, 2.9_real64, 5.2_real64, 6.8_real64, 9.1_real64]

  ! Inicialización: ningún acumulador debe comenzar con un valor indefinido
  sx = 0.0_real64
  sy = 0.0_real64
  sxx = 0.0_real64
  sxy = 0.0_real64

  ! Primera pasada: construye las cuatro sumas de mínimos cuadrados
  do i = 1, n
    sx = sx + x(i)          ! Suma de x
    sy = sy + y(i)          ! Suma de y
    sxx = sxx + x(i)*x(i)   ! Suma de x^2
    sxy = sxy + x(i)*y(i)   ! Suma de x*y
  end do

  ! Calcula los parámetros de la recta y evita una división por cero
  den = real(n, real64)*sxx - sx*sx
  if (abs(den) <= tiny(den)) error stop 'No se puede calcular la pendiente'

  a = (real(n, real64)*sxy - sx*sy) / den  ! Pendiente
  b = (sy - a*sx) / real(n, real64)        ! Intercepto

  ! Segunda pasada: calcula residuos, SSE y variación total SST
  sse = 0.0_real64
  sst = 0.0_real64
  do i = 1, n
    y_ajustada = a*x(i) + b                       ! Predicción de la recta
    sse = sse + (y(i) - y_ajustada)**2           ! Suma de residuos al cuadrado
    sst = sst + (y(i) - sy/real(n, real64))**2  ! Variación alrededor de la media
  end do

  ! R^2 solo está definido aquí si los valores de y no son todos iguales
  if (sst > 0.0_real64) then
    r2 = 1.0_real64 - sse/sst
  else
    r2 = 0.0_real64
  end if

  ! Presenta los resultados con etiquetas y formato controlado
  print '(A,I0)', 'N = ', n
  print '(A,F12.6)', 'pendiente a = ', a
  print '(A,F12.6)', 'intercepto b = ', b
  print '(A,F12.6)', 'R^2 = ', r2
end program ajuste_lineal

In [ ]:
!gfortran -std=f2008 -Wall -Wextra -fcheck=all -o ajuste_lineal ajuste_lineal.f90
!./ajuste_lineal

### Calidad del ajuste

El coeficiente de determinación se calcula como

$$R^2=1-\frac{\sum_i(y_i-\widehat y_i)^2}{\sum_i(y_i-\overline y)^2}. $$

Un valor cercano a 1 indica que la recta explica una fracción grande de la variación observada. Sin embargo, un $R^2$ alto no demuestra por sí solo que el modelo físico sea correcto. También debemos examinar los residuos, las unidades y el dominio de validez del modelo.

### Visualización del ajuste y los residuos

Fortran realiza el cálculo numérico. Python se usa aquí únicamente para visualizar los datos y evaluar el patrón de los residuos. Copiamos en `a` y `b` los valores producidos por el programa Fortran.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Lee las dos columnas del archivo de demostración
datos = np.loadtxt("datos_lineales.dat")
x = datos[:, 0]
y = datos[:, 1]

# Resultados obtenidos con el programa Fortran
a = 1.99
b = 1.04

# Construye la recta y calcula el residuo de cada punto
x_recta = np.linspace(x.min(), x.max(), 200)
y_recta = a*x_recta + b
residuos = y - (a*x + b)

# Crea dos paneles: datos con ajuste y residuos
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.scatter(x, y, color="navy", label="Datos")
ax1.plot(x_recta, y_recta, color="crimson",
         label=f"Ajuste: y = {a:.3f}x + {b:.3f}")
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.grid(alpha=0.3)
ax1.legend()

ax2.axhline(0.0, color="black", linewidth=1)
ax2.scatter(x, residuos, color="darkgreen")
ax2.set_xlabel("x")
ax2.set_ylabel("Residuo")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Ejercicio 3 — Reparar el ajuste

En el programa siguiente falta una actualización del acumulador. El código compila, pero produce una pendiente incorrecta. Añada la línea necesaria y compruebe que obtiene el mismo resultado del ejemplo completo.

In [ ]:
%%writefile ejercicio_ajuste.f90
program ejercicio_ajuste
  use iso_fortran_env, only: real64, iostat_end  ! Precisión y fin de archivo
  implicit none                                  ! Evita variables implícitas
  integer :: unidad, estado, n                   ! Control del archivo y contador
  real(real64) :: x, y, sx, sy, sxx, sxy, den, a, b  ! Datos, sumas y parámetros

  ! Inicializa el contador y los acumuladores antes de leer
  n = 0
  sx = 0.0_real64
  sy = 0.0_real64
  sxx = 0.0_real64
  sxy = 0.0_real64

  ! Abre el archivo de dos columnas generado anteriormente
  open(newunit=unidad, file='datos_lineales.dat', status='old', &
       action='read', iostat=estado)
  if (estado /= 0) error stop 'No fue posible abrir el archivo'

  ! Lee cada par (x,y) y construye las sumas necesarias
  do
    read(unidad, *, iostat=estado) x, y
    if (estado == iostat_end) exit
    if (estado /= 0) error stop 'Error de lectura'
    n = n + 1           ! Cuenta el par leído
    sx = sx + x         ! Acumula x
    sy = sy + y         ! Acumula y
    sxx = sxx + x*x     ! Acumula x^2
    ! TODO: falta actualizar uno de los acumuladores
  end do
  close(unidad)         ! Cierra el archivo después de la lectura

  ! Usa las sumas para calcular la recta y mostrar sus parámetros
  den = real(n, real64)*sxx - sx*sx
  a = (real(n, real64)*sxy - sx*sy) / den
  b = (sy - a*sx) / real(n, real64)
  print *, 'a =', a, 'b =', b
end program ejercicio_ajuste

In [ ]:
!gfortran -std=f2008 -Wall -Wextra -fcheck=all -o ejercicio_ajuste ejercicio_ajuste.f90
!./ejercicio_ajuste

## 4. Modelos que no parecen lineales

Algunas leyes físicas pueden transformarse para usar un ajuste lineal:

| Modelo original | Transformación | Recta resultante |
|---|---|---|
| $y=Ae^{kx}$ | $Y=\ln y$ | $Y=kx+\ln A$ |
| $y=Ax^n$ | $X=\ln x$, $Y=\ln y$ | $Y=nX+\ln A$ |
| $T=2\pi\sqrt{L/g}$ | $y=T^2$, $x=L$ | $y=(4\pi^2/g)x$ |

La transformación cambia la estructura de los errores. Si el error experimental es aditivo en la variable original, el ajuste linealizado no siempre equivale a minimizar los residuos originales. Para esta primera práctica usaremos la linealización, pero debemos declarar la transformación y revisar los residuos.

## 5. El péndulo simple

Para oscilaciones pequeñas, el período de un péndulo de longitud $L$ es

$$T=2\pi\sqrt{\frac{L}{g}}. $$

Al elevar al cuadrado obtenemos

$$T^2=\frac{4\pi^2}{g}L.$$

Por tanto, definimos

$$x=L\;(\mathrm{m}),\qquad y=T^2\;(\mathrm{s^2}).$$

Si ajustamos $y=ax+b$, la pendiente permite estimar

$$g=\frac{4\pi^2}{a}. $$

El modelo ideal predice $b=0$. En datos reales, un intercepto distinto de cero puede revelar error sistemático, limitaciones del modelo o problemas en las mediciones.

<div style="background-color:#eef7ee; border-left:5px solid #4caf50; padding:10px 15px; margin:10px 0;">
<b>Conversión de unidades:</b> el archivo contiene la longitud en centímetros, pero la estimación de <i>g</i> debe usar metros. En Fortran emplee <code>x = longitud_cm / 100.0_real64</code>. El período de una fila es <code>T = tiempo_s / real(n_osc, real64)</code>.
</div>

## 6. Cargar archivos en Google Colab

Los archivos guardados en Colab son temporales y desaparecen cuando termina la sesión. Para la tarea, cargue el Excel y su copia completada del script de limpieza con la celda siguiente.

In [ ]:
from google.colab import files

archivos_cargados = files.upload()
print("Archivos cargados:", list(archivos_cargados))

Después de completar los tres apartados TODO del script, puede ejecutarlo en Colab. Si cambió su nombre, modifique el comando.

In [ ]:
!python script_limpieza_pendulo_estudiantes.py
!ls -lh pendulo_limpio.dat informe_limpieza.xlsx

# Taller de la semana — Determinación de $g$ con un péndulo

## Propósito

Construir un flujo reproducible que empiece con 100 registros experimentales en Excel, documente la limpieza y termine con un ajuste lineal programado en Fortran 2008.

## Parte A. Limpieza de datos

1. Abra `100_mediciones_pendulo_minimos_cuadrados.xlsx` y revise la hoja `Datos_estudiantes`.
2. Ejecute el script de limpieza suministrado por el profesor.
3. Complete sus tres tareas: correcciones justificadas, elección del ángulo máximo y exclusiones manuales.
4. No excluya un dato únicamente porque se aleja de la tendencia. Toda corrección o exclusión debe apoyarse en la información experimental.
5. El script debe generar `pendulo_limpio.dat` e `informe_limpieza.xlsx`.

## Parte B. Programa en Fortran 2008

Escriba `ajuste_pendulo.f90`. El programa debe:

1. Abrir `pendulo_limpio.dat` y comprobar que el archivo se abrió correctamente.
2. Leer un número desconocido de filas mediante un ciclo y `iostat`.
3. Convertir cada longitud de centímetros a metros.
4. Calcular $T$ y construir $x=L$ y $y=T^2$.
5. Acumular $N$, $s_x$, $s_y$, $s_{xx}$ y $s_{xy}$.
6. Calcular la pendiente $a$ y el intercepto $b$.
7. Estimar $g=4\pi^2/a$.
8. Realizar una segunda lectura del archivo para calcular $R^2$.
9. Mostrar claramente $N$, $a$, $b$, $R^2$ y $g$, incluyendo unidades.
10. Escribir esos cinco valores, en ese orden y sin encabezado, en `resultados_ajuste.dat`.
11. Detectar al menos estos errores: archivo inexistente, menos de dos datos y denominador nulo.

Para estandarizar el archivo que revisará la celda PASS/FAIL, declare una unidad de salida entera y use al final del programa:

```fortran
open(newunit=unidad_salida, file='resultados_ajuste.dat', &
     status='replace', action='write')
write(unidad_salida, *) n, a, b, r2, g
close(unidad_salida)
```

Compile con advertencias y comprobaciones activadas:

```bash
gfortran -std=f2008 -Wall -Wextra -fcheck=all -o ajuste_pendulo ajuste_pendulo.f90
./ajuste_pendulo
```

## Parte C. Visualización en Python

Complete la celda para representar $T^2$ en función de $L$, la recta calculada con Fortran y los residuos. No use Python para volver a realizar el ajuste: los valores de `a` y `b` deben proceder de su programa Fortran.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Lee las cinco columnas producidas por el script de limpieza
datos = np.loadtxt("pendulo_limpio.dat")
longitud_m = datos[:, 1] / 100.0
numero_oscilaciones = datos[:, 3]
tiempo_total_s = datos[:, 4]

# TODO: construya las variables linealizadas x=L y y=T^2
periodo_s = _____
x = _____
y = _____

# Copie los parámetros calculados por su programa Fortran
a = _____
b = _____

# Ordena x para dibujar una línea continua de izquierda a derecha
orden = np.argsort(x)
y_ajustada = a*x + b
residuos = y - y_ajustada

# Primer panel: mediciones y recta. Segundo panel: residuos
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.scatter(x, y, s=25, color="navy", label="Mediciones")
ax1.plot(x[orden], y_ajustada[orden], color="crimson", label="Ajuste Fortran")
ax1.set_xlabel("Longitud L (m)")
ax1.set_ylabel(r"$T^2$ (s$^2$)")
ax1.grid(alpha=0.3)
ax1.legend()

ax2.axhline(0.0, color="black", linewidth=1)
ax2.scatter(x, residuos, s=25, color="darkgreen")
ax2.set_xlabel("Longitud L (m)")
ax2.set_ylabel("Residuo (s²)")
ax2.grid(alpha=0.3)

plt.tight_layout()
# Guarda la figura que se entregará con el informe
plt.savefig("ajuste_pendulo_y_residuos.png", dpi=200, bbox_inches="tight")
plt.show()

## Parte D. Informe de una página

Entregue un informe en PDF usando la plantilla LaTeX suministrada. Debe ocupar **una sola página**. Si el contenido excede una página, resuma las ideas; no reduzca la fuente ni cambie los márgenes.

Use como encabezado: **Taller 3 — Ajuste lineal y péndulo simple**, **Semana 3** y **Lenguajes: Fortran/Python**. Complete las seis secciones de la plantilla:

1. **Contexto físico.** En una o dos frases explique cómo el período de un péndulo permite estimar $g$ y por qué se requiere la aproximación de ángulo pequeño.
2. **Método.** Identifique el ajuste lineal por mínimos cuadrados y presente las relaciones centrales $T^2=(4\pi^2/g)L$ y $g=4\pi^2/a$. No copie el programa completo. Indique brevemente el criterio principal usado para limpiar los datos.
3. **Resultado.** Reporte $g$ frente al valor de referencia $9.81\,\mathrm{m/s^2}$ y su error relativo porcentual. En una frase adicional interprete el intercepto, $R^2$ y el patrón de los residuos. Copie también el resumen de la celda PASS/FAIL.
4. **Obstáculo o decisión de implementación.** Describa un problema concreto encontrado durante la limpieza, lectura del archivo o programación del ajuste, y explique cómo lo resolvió.
5. **Conclusión.** Escriba una o dos frases sobre lo aprendido y una limitación del resultado o del modelo.
6. **Uso de herramientas de IA.** Declare si utilizó IA, cuál herramienta y para qué. Si no utilizó ninguna, indíquelo explícitamente.

Calcule el error relativo porcentual mediante

$$\text{error relativo}=\left|\frac{g_{\mathrm{estimada}}-9.81}{9.81}\right|\times100\%. $$

La figura completa se entrega como archivo separado; no es obligatorio insertarla en el informe de una página.

## Entregables

- `informe_limpieza.xlsx`
- `pendulo_limpio.dat`
- `ajuste_pendulo.f90`
- `resultados_ajuste.dat`
- salida del programa con los resultados
- `ajuste_pendulo_y_residuos.png`
- informe de una página en PDF, elaborado con la plantilla LaTeX suministrada

## Criterios de revisión

Se revisará la trazabilidad de la limpieza, el uso correcto de Fortran 2008, la lectura robusta del archivo, la implementación de las fórmulas, las unidades y la interpretación física. Cada estudiante debe poder explicar su código y justificar sus decisiones.

### Lista de verificación antes de entregar

- [ ] Mi archivo `.dat` no tiene encabezado y contiene cinco columnas numéricas.
- [ ] Convertí centímetros a metros antes de estimar $g$.
- [ ] Inicialicé todos los acumuladores en cero.
- [ ] Mi programa lee hasta el final del archivo, sin asumir que siempre hay 100 filas.
- [ ] Compilé con `-Wall -Wextra -fcheck=all`.
- [ ] Reporté unidades y cifras razonables.
- [ ] Generé `resultados_ajuste.dat` con cinco valores y sin encabezado.
- [ ] Puedo explicar qué hace cada ciclo y cada acumulador.
- [ ] Mis exclusiones se basan en evidencia experimental y quedaron documentadas.

## Verificación final PASS/FAIL

Para agilizar la revisión, la tarea debe cerrar ejecutando esta celda. La verificación calcula valores de referencia directamente a partir de `pendulo_limpio.dat` y los compara con `resultados_ajuste.dat`. También comprueba que se haya generado la figura.

Un `PASS` indica que el resultado supera esa comprobación automática; no sustituye la revisión de la limpieza, la estructura del programa, las unidades ni la interpretación física.

In [ ]:
from pathlib import Path
import numpy as np

# Función auxiliar que imprime un veredicto uniforme para cada prueba
resultados_pruebas = []

def informar(nombre, condicion, detalle=""):
    estado = "PASS" if condicion else "FAIL"
    print(f"{estado:<4}  {nombre}{detalle}")
    resultados_pruebas.append(bool(condicion))
    return bool(condicion)

# Archivos que debe producir el flujo de trabajo completo
archivo_datos = Path("pendulo_limpio.dat")
archivo_resultados = Path("resultados_ajuste.dat")
archivo_figura = Path("ajuste_pendulo_y_residuos.png")

# Primer nivel: comprueba existencia antes de intentar leer
ok_datos = informar("Existe pendulo_limpio.dat", archivo_datos.exists())
ok_resultados = informar("Existe resultados_ajuste.dat", archivo_resultados.exists())
informar("Existe la figura", archivo_figura.exists() and archivo_figura.stat().st_size > 0 if archivo_figura.exists() else False)

# Segundo nivel: valida formatos y compara con un cálculo independiente
if ok_datos and ok_resultados:
    try:
        datos = np.atleast_2d(np.loadtxt(archivo_datos))
        resultados = np.loadtxt(archivo_resultados).reshape(-1)

        estructura_datos = datos.shape[1] == 5 and datos.shape[0] >= 2
        estructura_resultados = resultados.size == 5
        informar("Formato del archivo de datos", estructura_datos, f"  filas={datos.shape[0]}, columnas={datos.shape[1]}")
        informar("Formato del archivo de resultados", estructura_resultados, f"  valores={resultados.size}")

        if estructura_datos and estructura_resultados:
            # Reconstruye x=L y y=T^2 directamente desde los datos limpios
            longitud_m = datos[:, 1] / 100.0
            periodo_s = datos[:, 4] / datos[:, 3]
            x = longitud_m
            y = periodo_s**2
            n_ref = len(x)

            # Calcula los valores de referencia con las fórmulas del método
            sx = np.sum(x)
            sy = np.sum(y)
            sxx = np.sum(x*x)
            sxy = np.sum(x*y)
            den = n_ref*sxx - sx*sx
            a_ref = (n_ref*sxy - sx*sy) / den
            b_ref = (sy - a_ref*sx) / n_ref
            residuos_ref = y - (a_ref*x + b_ref)
            sse_ref = np.sum(residuos_ref**2)
            sst_ref = np.sum((y - np.mean(y))**2)
            r2_ref = 1.0 - sse_ref/sst_ref
            g_ref = 4.0*np.pi**2/a_ref

            # Compara, con tolerancia, cada resultado entregado por Fortran
            n_f, a_f, b_f, r2_f, g_f = resultados
            informar("Número de datos N", int(round(n_f)) == n_ref, f"  obtenido={n_f:g}, esperado={n_ref}")
            informar("Pendiente a", np.isclose(a_f, a_ref, rtol=1e-6, atol=1e-9), f"  obtenida={a_f:.8g}, esperada={a_ref:.8g}")
            informar("Intercepto b", np.isclose(b_f, b_ref, rtol=1e-6, atol=1e-9), f"  obtenido={b_f:.8g}, esperado={b_ref:.8g}")
            informar("Coeficiente R²", np.isclose(r2_f, r2_ref, rtol=1e-6, atol=1e-9), f"  obtenido={r2_f:.8g}, esperado={r2_ref:.8g}")
            informar("Gravedad g", np.isclose(g_f, g_ref, rtol=1e-6, atol=1e-9), f"  obtenida={g_f:.8g}, esperada={g_ref:.8g}")
            informar("Valor físico razonable de g", 7.0 < g_f < 12.0, f"  g={g_f:.5f} m/s²")
    except Exception as error:
        informar("No fue posible completar la verificación", False, f": {error}")

# Resumen que debe copiarse en el informe de una página
n_pass = sum(resultados_pruebas)
n_total = len(resultados_pruebas)
print(f"\nRESUMEN PASS/FAIL: {n_pass} de {n_total} pruebas pasaron.")